This Notebook demonstrates the concept of PyTorch datasets and dataloaders.

# DataSet

A dataset is a collection of data. In PyTorch, a dataset is represented by the `torch.utils.data.Dataset` class. A dataset can be created by subclassing the `Dataset` class and implementing the `__len__` and `__getitem__` methods. (__len__ returns the number of samples in the dataset, and __getitem__ returns a sample from the dataset given an index.)

# DataLoader
A dataloader is an iterator that provides batches of data from a dataset. In PyTorch, a dataloader is represented by the `torch.utils.data.DataLoader` class. A dataloader can be created by passing a dataset to the `DataLoader` class and specifying the batch size and other parameters.

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

In [55]:
class testds(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [56]:
# replicate ImageFolder dataset class, basic,

import os
from PIL import Image

class ImageFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        self._load_dataset()

    def _load_dataset(self):
        classes = sorted(entry.name for entry in os.scandir(self.root_dir) if entry.is_dir())
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
        self.idx_to_class = {idx: cls_name for cls_name, idx in self.class_to_idx.items()}

        for cls_name in classes:
            class_dir = os.path.join(self.root_dir, cls_name)
            for img_name in os.listdir(class_dir):
                img_path = os.path.join(class_dir, img_name)
                if os.path.isfile(img_path):
                    self.image_paths.append(img_path)
                    self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [57]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()])

ds = ImageFolderDataset(root_dir=r'data\test', transform=transform)